# Portfolio Assistant - Colab Runner

Run the cells in order. The same codebase works in VS Code - only the
environment/secret loading differs, and `Settings.load()` handles both.

**Required Colab secrets (userdata):** `LANGCHAIN_API_KEY` (optional),
`AI_GATEWAY_API_KEY`, `PINECONE_API_KEY`, `NOVITA_API_KEY`, `DATABASE_URL`,
`NGROK_AUTH_TOKEN` (optional).

## 1. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 2. Mount Google Drive and Configure Environment

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

# Settings.load() reads these env vars first; set them from Colab secrets
os.environ['LANGCHAIN_API_KEY'] = userdata.get('LangSmith_Api') or userdata.get('LANGCHAIN_API_KEY') or ''
os.environ['AI_GATEWAY_API_KEY'] = userdata.get('AI_GATEWAY_API_KEY') or ''
os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY') or ''
os.environ['NOVITA_API_KEY'] = userdata.get('NOVITA_API_KEY') or ''
os.environ['DATABASE_URL'] = userdata.get('DATABASE_URL') or ''

print('Environment configured')

## 3. Ingest Documents (run once, or when documents change)

In [ ]:
from config.settings import Settings
from embeddings.vercel_embeddings import VercelEmbeddings
from vectorstore.pinecone_store import PineconeVectorStoreProvider
from documents.loaders import DocumentLoaderService
from documents.chunker import DocumentChunker
from documents.ingestion import ingest_documents

settings = Settings.load()

embeddings = VercelEmbeddings(settings)
provider = PineconeVectorStoreProvider(settings)
provider.setup_index(dimension=embeddings.dimension)
vector_store = provider.get_vector_store(embeddings)

docs = DocumentLoaderService().load_pdf(
    '/content/drive/MyDrive/MyResume/ImranAliResumeForAiEnginriing.pdf'
)
chunker = DocumentChunker(settings.chunk_size, settings.chunk_overlap)
count = ingest_documents(docs, chunker, vector_store)
print(f'Indexed {count} chunks')

## 4. Start the API Server

In [ ]:
from main import app
from colab.server_utils import ServerManager

server = ServerManager(port=8000)
server.start(app)

## 5. Open the Chat UI

In [ ]:
# Option 1: Colab's built-in proxy (easiest, session-scoped)
from google.colab.output import eval_js
print('Chat UI:', eval_js('google.colab.kernel.proxyPort(8000)'))

# Option 2: Public URL via ngrok
# from google.colab import userdata
# server.connect_ngrok(userdata.get('NGROK_AUTH_TOKEN'))

## 6. Stop the Server

In [ ]:
server.stop()